In [13]:
!pip install google-genai sentence-transformers tqdm

  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
   ---------------------------------------- 0.0/950.8 kB ? eta -:--:--
   --------------------------------- ------ 786.4/950.8 kB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 950.8/950.8 kB 4.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 10.8 MB/s eta 0:00:00
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
   ---------------------------------------- 0.0/596.4 kB ? eta -:--:--
   ---------------------------------------- 596.4/596.4 kB 9.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/123.0 MB 11.8 MB/s eta 0:00:11
   - -------------------------------------- 3.4/123.0 MB 8.6 MB/s eta 0:00:14
   - -------------------------------------- 5.8/123.0 MB 9.5 MB/s eta 0:00:13
   -- ----------------------

In [13]:
!pip install wandb

In [10]:
API_KEY="AQ.Ab8RN6IM-xrt6A2A3p1rv96kqt9OqNe3ih3iZlBgseAzrhlbcA"

In [14]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Mad\_netrc.
wandb: Currently logged in as: oktmadmom (oktmadmom-innovasoft) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [11]:
import os
import json
import re
import time
from google import genai
from google.genai import types
from sentence_transformers import SentenceTransformer
import numpy as np
from tqdm import tqdm

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", API_KEY)

SEED_INSTRUCTIONS = [
    "Привет! Какое у вас есть альтернативное молоко?",
    "Как к вам пройти от метро?",
    "У вас можно посидеть с ноутбуком, есть розетки?",
    "Что такое флэт уайт и чем он отличается от капучино?",
    "Порекомендуй что-нибудь бодрящее, но без молока.",
    "Я пролил ваш раф на ноутбук, что мне делать?!",
    "Жду свой лонг блэк уже 15 минут, сколько можно?",
    "Помоги выбрать десерт к горькому эспрессо.",
    "У вас можно заказать кофе на зерне собственной обжарки?",
    "Есть ли у вас скидки, если прийти со своей термокружкой?"
]


def basic_filter(raw_data):
    clean_list = []
    bot_patterns = [
        r"как языковая модель", r"я всего лишь ии", r"не имею физического тела", 
        r"как искусственный интеллект", r"я не пью кофе"
    ]
    
    print("\n[Шаг 2/3] Запуск базовой фильтрации качества...")
    for item in raw_data:
        if not isinstance(item, dict) or "instruction" not in item or "response" not in item:
            continue
            
        inst = item["instruction"].strip()
        resp = item["response"].strip()
        
        if len(inst.split()) < 2 or len(resp.split()) < 5:
            continue  
        combined_text = (inst + " " + resp).lower()
        if any(re.search(pattern, combined_text) for pattern in bot_patterns):
            continue
            
        clean_list.append({"instruction": inst, "response": resp})
        
    print(f"-> После базовой фильтрации осталось: {len(clean_list)} из {len(raw_data)}")
    return clean_list


def semantic_deduplication(data, threshold=0.85):
    print("\n[Шаг 3/3] Запуск семантической дедупликации (SentenceTransformers)...")
    if not data:
        return data

    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    
    instructions = [item["instruction"] for item in data]
    embeddings = model.encode(instructions, show_progress_bar=True)

    embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
    similarity_matrix = np.dot(embeddings, embeddings.T)
    
    keep_indices = []
    dropped_count = 0
    
    for i in range(len(data)):
        is_duplicate = False
        for j in keep_indices:
            if similarity_matrix[i, j] > threshold:
                is_duplicate = True
                dropped_count += 1
                break
        if not is_duplicate:
            keep_indices.append(i)
            
    final_data = [data[idx] for idx in keep_indices]
    print(f"-> Удалено семантических дубликатов: {dropped_count}")
    print(f"-> Финальный размер датасета: {len(final_data)}")
    return final_data


def main():
    if not GEMINI_API_KEY:
        raise ValueError("API-ключ пустой. Пожалуйста, проверьте переменную GEMINI_API_KEY.")

    client = genai.Client(api_key=GEMINI_API_KEY)
    
    system_instruction = (
        "Ты — опытный дата-инженер ИИ. Твоя задача — сгенерировать обучающий датасет для "
        "LLM-виджета кофейни в формате Instruction-Response. Ответы должны строго имитировать стиль "
        "'Дружелюбный Бариста-Эксперт': теплый тон, легкий кофейный сленг (зерно, альтернатива, "
        "кислинка/плотность, крафт), но при этом давать четкий ответ по ситуации. Избегай канцеляризмов "
        "и фраз вроде 'Я текстовый ИИ'. Выдавай строго валидный JSON список объектов."
    )
    
    raw_dataset = []
    
    print("[Шаг 1/3] Запуск пакетной генерации датасета через Gemini 3.1 Flash-Lite...")
    
    for batch in range(25):
        print(f"Генерация пакета {batch + 1}/25...")
        
        prompt = f"""
        Используя эти базовые примеры в качестве вдохновения: {json.dumps(SEED_INSTRUCTIONS, ensure_ascii=False)}
        
        Сгенерируй ровно 10 уникальных пар "instruction" (разнообразные вопросы, просьбы, жалобы клиентов кофейни) 
        и "response" (ответы бариста). Обязательно охвати темы: меню, зерно, растительное молоко, атмосфера, 
        жалобы на скорость, казусы (пролил кофе). Не повторяй идеи из прошлых шагов.
        
        Выходной формат должен быть СТРОГО JSON-массивом (без разметки ```json ... ```):
        [
          {{"instruction": "текст", "response": "текст"}},
          ...
        ]
        """
        
        try:
            response = client.models.generate_content(
                model='gemini-3.1-flash-lite',
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    temperature=0.8,
                    response_mime_type="application/json" 
                ),
            )
            
            batch_data = json.loads(response.text)
            if isinstance(batch_data, list):
                raw_dataset.extend(batch_data)
            
            time.sleep(5)
            
        except Exception as e:
            print(f"Ошибка на шаге {batch + 1}: {e}")
            time.sleep(5)
            continue
            
    print(f"-> Успешно получено {len(raw_dataset)} сырых примеров от API.")

    if not raw_dataset:
        print("Датасет пуст. Завершение работы.")
        return

    filtered_data = basic_filter(raw_dataset)
    final_dataset = semantic_deduplication(filtered_data, threshold=0.88)
    
    output_path = "data/coffee_dataset.jsonl"
    os.makedirs("data", exist_ok=True)
    
    with open(output_path, "w", encoding="utf-8") as f:
        for item in final_dataset:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
            
    print(f"\n[УСПЕХ] Финальный датасет сохранен в '{output_path}'")
    print("Вы готовы переходить к Шагу 2 — Fine-tuning в Colab!")


if __name__ == "__main__":
    main()

[Шаг 1/3] Запуск пакетной генерации датасета через Gemini 3.1 Flash-Lite...
Генерация пакета 1/25...
Генерация пакета 2/25...
Генерация пакета 3/25...
Генерация пакета 4/25...
Генерация пакета 5/25...
Генерация пакета 6/25...
Ошибка на шаге 6: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Генерация пакета 7/25...
Генерация пакета 8/25...
Генерация пакета 9/25...
Генерация пакета 10/25...
Генерация пакета 11/25...
Генерация пакета 12/25...
Генерация пакета 13/25...
Генерация пакета 14/25...
Генерация пакета 15/25...
Генерация пакета 16/25...
Генерация пакета 17/25...
Генерация пакета 18/25...
Генерация пакета 19/25...
Генерация пакета 20/25...
Генерация пакета 21/25...
Генерация пакета 22/25...
Генерация пакета 23/25...
Ошибка на шаге 23: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. 

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

-> Удалено семантических дубликатов: 64
-> Финальный размер датасета: 165

[УСПЕХ] Финальный датасет сохранен в 'data/coffee_dataset.jsonl'
Вы готовы переходить к Шагу 2 — Fine-tuning в Colab!


In [ ]:
import json
import os
import re
import time
from google import genai
from google.genai import types
from sentence_transformers import SentenceTransformer
import numpy as np

output_path = "data/coffee_dataset.jsonl"
existing_data = []

if os.path.exists(output_path):
    with open(output_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                existing_data.append(json.loads(line.strip()))
    print(f"Загружено существующих примеров из файла: {len(existing_data)}")
else:
    print("Старый файл не найден, будет создан новый датасет.")

client = genai.Client(api_key=GEMINI_API_KEY)

system_instruction = (
    "Ты — опытный дата-инженер ИИ. Твоя задача — сгенерировать обучающий датасет для "
    "LLM-виджета кофейни в формате Instruction-Response. Ответы должны строго имитировать стиль "
    "'Дружелюбный Бариста-Эксперт': теплый тон, легкий кофейный сленг (зерно, альтернатива, "
    "кислинка/плотность, крафт), но при этом давать четкий ответ по ситуации. Избегай канцеляризмов "
    "и фраз вроде 'Я текстовый ИИ'. Выдавай строго валидный JSON список объектов."
)

new_raw_dataset = []
print("Запуск дополнительных 7 запросов через Gemini 3.1 Flash-Lite...")

for batch in range(7):
    print(f"Генерация дополнительного пакета {batch + 1}/7...")
    
    prompt = f"""
    Используя эти базовые примеры в качестве вдохновения: {json.dumps(SEED_INSTRUCTIONS, ensure_ascii=False)}
    
    Сгенерируй ровно 10 уникальных пар "instruction" (разнообразные вопросы, просьбы, жалобы клиентов кофейни) 
    и "response" (ответы бариста). Обязательно охвати темы: меню, зерно, растительное молоко, атмосфера, 
    жалобы на скорость, казусы (пролил кофе). Придумай новые ситуации, не повторяй избитые темы.
    
    Выходной формат должен быть СТРОГО JSON-массивом (без разметки ```json ... ```):
    [
      {{"instruction": "текст", "response": "текст"}},
      ...
    ]
    """
    
    try:
        response = client.models.generate_content(
            model='gemini-3.1-flash-lite',
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.85,  
                response_mime_type="application/json" 
            ),
        )
        
        batch_data = json.loads(response.text)
        if isinstance(batch_data, list):
            new_raw_dataset.extend(batch_data)
        
        time.sleep(5)
        
    except Exception as e:
        print(f"Ошибка на шаге {batch + 1}: {e}")
        time.sleep(5)
        continue

print(f"-> Получено новых сырых примеров: {len(new_raw_dataset)}")

full_dataset = existing_data + new_raw_dataset
print(f"Всего элементов для полной перепроверки: {len(full_dataset)}")

bot_patterns = [
    r"как языковая модель", r"я всего лишь ии", r"не имею физического тела", 
    r"как искусственный интеллект", r"я не пью кофе"
]

clean_list = []
for item in full_dataset:
    if not isinstance(item, dict) or "instruction" not in item or "response" not in item:
        continue
    inst = item["instruction"].strip()
    resp = item["response"].strip()
    
    if len(inst.split()) < 2 or len(resp.split()) < 5:
        continue  
    combined_text = (inst + " " + resp).lower()
    if any(re.search(pattern, combined_text) for pattern in bot_patterns):
        continue
        
    clean_list.append({"instruction": inst, "response": resp})

print(f"После фильтрации качества осталось: {len(clean_list)}")

if clean_list:
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    
    instructions = [item["instruction"] for item in clean_list]
    embeddings = model.encode(instructions, show_progress_bar=True)
    embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
    similarity_matrix = np.dot(embeddings, embeddings.T)
    
    keep_indices = []
    dropped_count = 0
    
    for i in range(len(clean_list)):
        is_duplicate = False
        for j in keep_indices:
            if similarity_matrix[i, j] > 0.88: 
                is_duplicate = True
                dropped_count += 1
                break
        if not is_duplicate:
            keep_indices.append(i)
            
    final_dataset = [clean_list[idx] for idx in keep_indices]
    print(f"-> Удалено дубликатов при полной проверке: {dropped_count}")
    print(f"-> Итоговый чистый размер датасета: {len(final_dataset)}")

    os.makedirs("data", exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        for item in final_dataset:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
            
    print(f"\n Обновленный датасет успешно сохранен в '{output_path}'")
else:
    print("Нет данных для сохранения.")

Загружено существующих примеров из файла: 165
Запуск дополнительных 7 запросов через Gemini 3.1 Flash-Lite...
Генерация дополнительного пакета 1/7...
Генерация дополнительного пакета 2/7...
Ошибка на шаге 2: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Генерация дополнительного пакета 3/7...
Генерация дополнительного пакета 4/7...
Генерация дополнительного пакета 5/7...
Генерация дополнительного пакета 6/7...
Генерация дополнительного пакета 7/7...
-> Получено новых сырых примеров: 60
Всего элементов для полной перепроверки: 225
После фильтрации качества осталось: 225


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

-> Удалено дубликатов при полной проверке: 20
-> Итоговый чистый размер датасета: 205

[УСПЕХ] Обновленный датасет успешно сохранен в 'data/coffee_dataset.jsonl'


In [15]:
import os
import json
import wandb

output_path = "data/coffee_dataset.jsonl"

if 'final_dataset' not in locals():
    print("Переменные не найдены в памяти, считываем данные напрямую из сохраненного файла...")
    final_dataset = []
    with open(output_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                final_dataset.append(json.loads(line.strip()))

wandb.login()

run = wandb.init(
    project="coffee-bot-finetuning", 
    job_type="data-preprocessing",
    notes="Логирование уже сгенерированного и очищенного датасета"
)

metrics = {"final_clean_dataset_size": len(final_dataset)}
if 'full_dataset' in locals(): metrics["raw_elements_combined"] = len(full_dataset)
if 'clean_list' in locals(): metrics["after_basic_filter"] = len(clean_list)
if 'dropped_count' in locals(): metrics["semantic_duplicates_removed"] = dropped_count

wandb.log(metrics)

artifact = wandb.Artifact(
    name="coffee_dataset", 
    type="dataset",
    description="Финальный очищенный Instruction-Response датасет для кофейни"
)
artifact.add_file(local_path=output_path)
run.log_artifact(artifact)

run.finish()
print(" Этап обработки данных успешно залогирован в Weights & Biases")

wandb: Detected [google.genai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


after_basic_filter,▁
final_clean_dataset_size,▁
raw_elements_combined,▁
semantic_duplicates_removed,▁
after_basic_filter,225
final_clean_dataset_size,205
raw_elements_combined,225
semantic_duplicates_removed,20


 Этап обработки данных успешно залогирован в Weights & Biases
